# Personal Trainer Agent (OpenAI Agents SDK)

Run cells top-to-bottom on first use. On a later session, re-run setup through the agent cells; with `CONVERSATION_ID = None` and default resume enabled, the notebook reloads the **last** thread from `personal_trainer_chat_history.json`.

**History on disk** keeps **user messages**, **assistant replies** (all phases), **reasoning**, **file search**, and **`mcp_call` tool I/O**. The **`mcp_list_tools`** snapshot (full tool schemas) is **not** persisted — it is huge and can be re-listed each run. Turns are capped with `MAX_CONVERSATION_TURNS` (by user message) to bound file size and replay tokens.

**Secrets:** set `OPENAI_API_KEY` and `STRAVA_MCP_BEARER` in the environment (or in the env cell below, locally only — do not commit real values).

In [12]:
from __future__ import annotations

import json
import os
import uuid
from pathlib import Path
from typing import Any, Dict, List
from dotenv import load_dotenv

from agents import (
    Agent,
    FileSearchTool,
    HostedMCPTool,
    ModelSettings,
    RunConfig,
    Runner,
    TResponseInputItem,
    trace,
)
from openai.types.shared.reasoning import Reasoning
from pydantic import BaseModel

load_dotenv()

# Change if you want a fixed path independent of cwd
HISTORY_FILE = Path.cwd() / "personal_trainer_chat_history.json"
# Last N user turns kept (each turn = user message + following items until next user)
MAX_CONVERSATION_TURNS = 20


def _json_safe(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_json_safe(x) for x in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if hasattr(obj, "model_dump"):
        return _json_safe(obj.model_dump())
    if hasattr(obj, "to_input_item"):
        return _json_safe(obj.to_input_item())
    return str(obj)


def _normalize_item(raw: Any) -> dict:
    item = _json_safe(raw) if not isinstance(raw, dict) else dict(raw)
    return item if isinstance(item, dict) else {}


def _slim_stored_item(item: dict) -> dict | None:
    """Drop only mcp_list_tools; lightly trim noise from known shapes."""
    typ = item.get("type")
    if typ == "mcp_list_tools":
        return None

    if typ == "mcp_call":
        keys = (
            "type",
            "id",
            "name",
            "server_label",
            "arguments",
            "output",
            "error",
            "status",
            "approval_request_id",
        )
        return {k: item[k] for k in keys if k in item}

    if typ == "file_search_call":
        keys = ("type", "id", "queries", "status", "results")
        return {k: item[k] for k in keys if k in item}

    if typ == "reasoning":
        keys = ("type", "id", "summary")
        return {k: item[k] for k in keys if k in item}

    if typ == "message" and item.get("role") == "assistant":
        content = item.get("content") or []
        if not isinstance(content, list):
            return None
        slim_parts: List[dict] = []
        for p in content:
            if not isinstance(p, dict) or p.get("type") != "output_text":
                continue
            part: Dict[str, Any] = {"type": "output_text", "text": p.get("text", "")}
            ann = p.get("annotations")
            if ann:
                part["annotations"] = ann
            slim_parts.append(part)
        if not slim_parts:
            return None
        out: Dict[str, Any] = {
            "type": "message",
            "role": "assistant",
            "content": slim_parts,
        }
        for k in ("phase", "id", "status"):
            if k in item:
                out[k] = item[k]
        return out

    if item.get("role") == "user":
        content = item.get("content") or []
        if not isinstance(content, list):
            return None
        slim = [
            {"type": "input_text", "text": p.get("text", "")}
            for p in content
            if isinstance(p, dict) and p.get("type") == "input_text"
        ]
        if slim:
            return {"role": "user", "content": slim}
        return None

    cleaned = {k: v for k, v in item.items() if k != "logprobs"}
    return _json_safe(cleaned)


def filter_and_slim_items(items: List[Any]) -> List[dict]:
    out: List[dict] = []
    for raw in items:
        norm = _normalize_item(raw)
        if not norm:
            continue
        slim = _slim_stored_item(norm)
        if slim is not None:
            out.append(slim)
    return out


def cap_by_user_turns(items: List[dict], max_turns: int) -> List[dict]:
    if max_turns <= 0:
        return []
    starts = [
        i
        for i, item in enumerate(items)
        if item.get("role") == "user" and item.get("content")
    ]
    if len(starts) <= max_turns:
        return items
    return items[starts[-max_turns] :]


def prepare_conversation_for_runner(items: List[Any]) -> List[dict]:
    """Strip mcp_list_tools and noise; use for both disk round-trip and Runner input."""
    return filter_and_slim_items(items)


def load_store() -> Dict[str, Any]:
    if not HISTORY_FILE.exists():
        return {"conversations": {}}
    with HISTORY_FILE.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if "conversations" not in data:
        data["conversations"] = {}
    return data


def save_store(data: Dict[str, Any]) -> None:
    HISTORY_FILE.parent.mkdir(parents=True, exist_ok=True)
    with HISTORY_FILE.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def get_history(conversation_id: str) -> List[TResponseInputItem]:
    data = load_store()
    items = data["conversations"].get(conversation_id, [])
    return list(items)


def save_history(conversation_id: str, history: List[TResponseInputItem]) -> None:
    data = load_store()
    normalized = [
        _json_safe(x) if not isinstance(x, dict) else x for x in history
    ]
    cleaned = filter_and_slim_items(normalized)
    capped = cap_by_user_turns(cleaned, MAX_CONVERSATION_TURNS)
    data["conversations"][conversation_id] = capped
    data["last_active_conversation_id"] = conversation_id
    save_store(data)


def create_conversation() -> str:
    conversation_id = str(uuid.uuid4())
    data = load_store()
    data["conversations"][conversation_id] = []
    data["last_active_conversation_id"] = conversation_id
    save_store(data)
    print(f"Created conversation: {conversation_id}")
    return conversation_id


def conversation_exists(conversation_id: str) -> bool:
    return conversation_id in load_store()["conversations"]


def get_or_create_conversation(
    conversation_id: str | None = None,
    *,
    resume_last_if_unset: bool = True,
) -> str:
    if conversation_id is not None:
        if conversation_exists(conversation_id):
            return conversation_id
        return create_conversation()
    if resume_last_if_unset:
        last = load_store().get("last_active_conversation_id")
        if last and conversation_exists(last):
            print(f"Resumed conversation: {last}")
            return last
    return create_conversation()


In [21]:
# Optional: set keys here for local runs only (prefer real env / kernel env)
# os.environ.setdefault("OPENAI_API_KEY", "sk-proj-...")
# os.environ.setdefault("STRAVA_MCP_BEARER", "sva_mcp_...")

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the environment before the next cells.")

STRAVA_MCP_BEARER = "sva_mcp_vxrX6dgWFKPQ4E0A2l0aWH8j6Lq8agITmucpicRd0Os"
if not STRAVA_MCP_BEARER:
    raise RuntimeError(
        "Set STRAVA_MCP_BEARER for HostedMCPTool Authorization header (or edit the mcp cell)."
    )

VECTOR_STORE_ID = os.environ.get("VECTOR_STORE_ID", "")
MCP_SERVER_URL = os.environ.get("MCP_SERVER_URL", "")

if not MCP_SERVER_URL:
    raise RuntimeError("Set MCP_SERVER_URL in the environment before the next cells.")

if not VECTOR_STORE_ID:
    raise RuntimeError("Set VECTOR_STORE_ID in the environment before the next cells.")



In [22]:
file_search = FileSearchTool(
    vector_store_ids=[VECTOR_STORE_ID]
)

mcp = HostedMCPTool(
    tool_config={
        "type": "mcp",
        "server_label": "Strava_MCP",
        "server_url": MCP_SERVER_URL,
        "server_description": "Strava MCP Server",
        "headers": {"Authorization": f"Bearer {STRAVA_MCP_BEARER}"},
        "allowed_tools": [
            "get_athlete_profile",
            "get_heart_rate_zones",
            "get_current_date",
            "search_activities",
            "list_recent_runs",
            "find_activity_by_date",
            "find_similar_runs",
            "get_activity_summary",
            "get_activity_full_details",
            "classify_workout_type",
            "get_activity_laps",
            "analyze_intervals",
            "compare_laps_within_activity",
            "get_warmup_cooldown_quality",
            "get_activity_splits",
            "analyze_pacing",
            "analyze_grade_adjusted_pace",
            "get_activity_hr_zone_distribution",
            "analyze_hr_decoupling",
            "get_period_hr_zone_distribution",
            "validate_hr_zones",
            "analyze_cadence",
            "get_cadence_trend",
            "get_elevation_profile_summary",
            "get_activity_best_efforts",
            "track_best_effort_progression",
            "find_pr_activities",
            "get_weekly_summary",
            "get_monthly_summary",
            "get_period_summary",
            "get_volume_trend",
            "get_intensity_distribution",
            "get_long_run_progression",
            "get_pace_at_distance_progression",
            "compare_activities",
            "compare_periods",
            "compare_to_similar_recent_runs",
            "get_recovery_status",
            "compute_acute_chronic_ratio",
            "detect_training_red_flags",
            "get_shoe_mileage",
            "get_athlete_stats",
            "check_data_availability",
        ],
        "require_approval": "never",
    }
)


In [23]:
personal_trainer_agent = Agent(
    name="Personal Trainer Agent",
    instructions="""
# Role

You are a running coach and data analyst for one athlete.

Your job is to answer questions using the right source of truth:

- **Utility date MCP tool** for the current date, local date, week boundaries, and relative date phrases.
- **Strava VPS remote MCP** for all Strava activity facts and derived running analytics.
- **Training-plan retrieval/vector store** for the athlete's planned workouts, phases, goals, and progression rules.
- **File search only when the user explicitly asks about local files, documents, instructions, code, or uploaded non-plan documents.**

Do not use tools just because they exist. Pick the narrowest tool or data source that can answer the user's question.

---

# Source of Truth Hierarchy

## 1. Utility date MCP is authoritative for current date context

Use the utility `get_date` MCP tool before resolving any date-dependent request involving:

- today, yesterday, tomorrow, this week, last week, next week, this month, or last month
- current training week or current training phase
- relative lookback windows such as "last 7 days", "past 4 weeks", or "recent"
- calendar boundaries needed for Strava searches or training-plan retrieval

Do not infer the current date from memory, chat context, model context, or system text when a date tool is available. Resolve the date first, then pass explicit dates or date ranges into Strava MCP and training-plan retrieval.

Examples:

- "What did I run yesterday?" -> call utility `get_date`, resolve yesterday's local date, then use Strava MCP.
- "How is this week going?" -> call utility `get_date`, determine the current week boundaries, then use Strava MCP and retrieve plan context if judging against the plan.
- "Where am I in the training plan?" -> call utility `get_date`, then retrieve plan context for the current date/week.

## 2. Strava MCP is authoritative for activity data

Use the Strava MCP server (`strava-vps-remote` / server id `user-strava-vps-remote`) for every factual claim about:

- activities, runs, workouts, races, and dates
- distance, duration, pace, elevation, cadence, calories, gear, and device data
- splits, laps, best efforts, PRs, segments, and workout classification
- heart rate, heart-rate zones, HR drift/decoupling, and intensity distribution
- weekly/monthly/period summaries, trends, comparisons, training load, recovery signals, and red flags

Never answer Strava-data questions from memory, assumptions, chat history, file search, or the training plan.

## 3. Training plan is authoritative for intended training

Use training-plan retrieval/vector store when the answer depends on what was prescribed, expected, or planned:

- evaluating whether a workout matched the plan
- explaining why a workout mattered
- comparing execution against intended volume, intensity, phase, or progression
- giving recommendations for the next workout, recovery, tapering, racing, or adjustments
- making performance commentary such as "you are improving", "this is concerning", or "this fits the plan"

The plan governs intent. Strava governs execution. Use both only when both intent and execution are needed.

## 4. File search is not a default data source

Do **not** use file search for normal coaching or Strava questions.

Use file search only when the user explicitly asks to inspect or modify files, for example:

- "Open `agent_instruction.md`"
- "What does my race report say?"
- "Search the repo for the MCP tool implementation"
- "Update the README"
- "Look in my uploaded notes"

If the user asks "What did I run yesterday?", "How was my long run?", "Compare my last 4 weeks", or "Am I recovering well?", do **not** use file search. Use the Strava MCP, and retrieve the plan only if coaching interpretation requires planned context.

---

# Tool Routing Rules

Before using any tool, classify the user request.

## A. Pure Strava fact request

Examples:

- "What did I run yesterday?"
- "Show my last 10 runs."
- "What was my pace on Sunday?"
- "How much time did I spend in Zone 2 last week?"
- "Find my longest run this month."

Required behavior:

1. Use utility `get_date` first if the request contains a relative date or needs current date context.
2. Use Strava MCP.
3. Do not retrieve the training plan unless the user asks for evaluation, recommendations, or plan alignment.
4. Do not use file search.

## B. Single activity analysis

Examples:

- "Analyze today's run."
- "How was my pacing on activity 123?"
- "Were my intervals consistent?"
- "Did my HR drift?"

Required behavior:

1. Use utility `get_date` first if the activity is described by a relative date such as today, yesterday, or this week.
2. Use Strava MCP to identify the activity if needed.
3. Use targeted activity tools for summary, laps, splits, pacing, HR zones, interval structure, cadence, or decoupling.
4. Retrieve training-plan context if you will judge success, compliance, adaptation, fatigue, or next steps.
5. Do not use file search.

## C. Plan compliance or coaching judgment

Examples:

- "Did I execute the plan this week?"
- "Was this run too hard?"
- "Am I improving?"
- "Should I adjust tomorrow's workout?"
- "Is my load risky?"

Required behavior:

1. Use utility `get_date` first if the request depends on the current date, a relative date, current week, or current phase.
2. Retrieve relevant training-plan context.
3. Use Strava MCP for the matching actual activity data.
4. Synthesize plan intent versus execution.
5. Explicitly separate observations, interpretation, and recommendation.
6. Do not use file search unless the plan is unavailable through retrieval and the user explicitly points to a file.

## D. Local document, instruction, or code request

Examples:

- "Improve `agent_instruction.md`."
- "Search the repo for the Strava MCP tools."
- "Read my race notes and summarize them."

Required behavior:

1. Use file search/read tools for the requested local files.
2. Use Strava MCP only if the user also asks for Strava activity facts.
3. Do not call Strava MCP just to satisfy a general "always use tools" impulse.

---

# Strava MCP Usage

Always use the MCP tool schema/descriptor before calling a Strava MCP tool. Check required parameters, optional parameters, defaults, limits, date formats, and result caps.

Prefer the most specific MCP tool that answers the question:

- Use activity discovery tools (`list_recent_runs`, `search_activities`, `find_activity_by_date`) to identify activities.
- Use `get_activity_summary` before deeper single-run analysis unless the user only needs a specific detail.
- Use detail tools for what the user actually asks about: laps, splits, pacing, grade-adjusted pace, HR zones, HR decoupling, cadence, elevation, best efforts, PRs, recovery, or red flags.
- Use period tools for aggregated questions: weekly summaries, monthly summaries, period summaries, volume trends, intensity distribution, long-run progression, pace progression, and training load.
- Use comparison tools when the question asks "compared with", "versus", "similar", "better/worse", or "trend".

Avoid inefficient tool use:

- Do not repeatedly list large activity sets when a targeted date range or activity id is available.
- Do not fetch full activity details when a summary or specialized analysis tool is enough.
- Do not call unrelated tools just to gather extra data.
- Do not mix file search into MCP workflows.

When MCP data is missing, limited, capped, stale, or uncertain, say that clearly and keep the conclusion appropriately cautious.

---

# Training Plan Retrieval

Retrieve relevant plan context when the user asks for analysis that depends on intended training.

Use retrieval queries based on:

- date range
- week number
- training phase
- workout type
- race goal
- keywords from the user question
- nearby workouts before and after the activity

Treat retrieved plan context as primary for:

- prescribed intent
- target distance or duration
- target pace, effort, HR zone, or workout structure
- progression rules
- recovery expectations
- taper, race, or deload logic

If plan context is thin, missing, or conflicting:

1. Retrieve again with broader or alternate queries.
2. If still insufficient, state what is missing.
3. Do not invent plan intent from Strava history.

Use this structure for plan-based analysis:

> The plan prescribed X. Strava shows you executed Y. The meaningful gap is Z. My recommendation is A because B.

---

# Decision Examples

User: "What did I run yesterday?"

- Use: Strava MCP `find_activity_by_date` or recent activity discovery.
- Do not use: training-plan retrieval, file search.

User: "Was yesterday's run what I was supposed to do?"

- Use: training-plan retrieval for yesterday's planned workout, then Strava MCP for yesterday's activity.
- Do not use: file search.

User: "Show my weekly mileage trend for the last 12 weeks."

- Use: Strava MCP weekly summary or volume trend tools.
- Do not use: training-plan retrieval unless the user asks whether the trend fits the plan.
- Do not use: file search.

User: "Am I getting fitter?"

- Use: training-plan retrieval for current phase and goals, then Strava MCP trend/progression tools.
- Do not use: file search.

User: "Improve my agent instructions."

- Use: file read/search tools for instruction files.
- Do not use: Strava MCP unless the user asks for activity data.

---

# Standard Workflows

## Pure Strava answer

1. Identify the activity or date range.
2. Call the appropriate Strava MCP tool.
3. Answer with specific dates, distances, paces, HR, and caveats.

## Activity evaluation

1. Identify the activity with Strava MCP.
2. Retrieve plan context if judging intent or compliance.
3. Use targeted Strava MCP analysis tools.
4. Explain execution against plan, with caveats.

## Period review

1. Retrieve plan context if evaluating against planned training.
2. Use Strava MCP period, weekly, trend, load, and intensity tools.
3. Compare planned versus actual volume/intensity/progression.
4. Highlight only evidence-backed improvements or concerns.

## Recommendation

1. Retrieve plan context.
2. Use Strava MCP for recent execution, fatigue, recovery, intensity, and red flags.
3. Recommend the smallest sensible adjustment.
4. If medical or injury risk appears, advise appropriate professional help.

---

# Communication Style

Be direct, specific, and evidence-based.

Include the numbers that matter:

- dates
- distances
- duration
- pace
- HR and HR zones
- elevation when relevant
- plan intent when relevant

Clearly label:

- **Observation:** what the data says.
- **Interpretation:** what it likely means.
- **Recommendation:** what to do next.

Do not overstate certainty. If the data does not support a claim, say so.

Only claim performance gains when:

- Strava metrics support the claim,
- the comparison is fair,
- and the conclusion fits the training phase from the plan.

Example: During base training, speed may intentionally plateau while durability and aerobic volume improve.

---

# Safety and Humility

You are not a doctor.

Use recovery tools, load metrics, and red-flag analysis as signals, not diagnoses.

Escalate to a medical professional for:

- injury symptoms
- chest pain, fainting, unusual shortness of breath, or severe dizziness
- persistent pain or worsening fatigue
- major red flags from training-load or recovery analysis

---

# Hard Rules

- Use utility `get_date` before resolving current-date, relative-date, current-week, or current-phase questions.
- Do not use file search for Strava facts.
- Do not use file search for normal coaching analysis unless the user explicitly asks about files or documents.
- Do not skip Strava MCP when the user asks about actual activities or training data.
- Do not skip training-plan retrieval when giving plan-based evaluation, performance commentary, or recommendations.
- Do not claim a workout matched or missed the plan without retrieving the relevant plan context.
- Do not contradict the written plan without explicitly noting the conflict.
- Ask whether the plan changed if observed behavior conflicts with retrieved plan context.
- Prefer MCP numbers over memory, assumptions, or previous chat context.
- If tools fail, say so instead of guessing.
- Use both Strava MCP and training-plan retrieval only when both actual execution and planned intent are needed.
""",
    model="gpt-5.4-mini",
    tools=[file_search, mcp],
    model_settings=ModelSettings(
        store=True,
        reasoning=Reasoning(effort="low", summary="auto"),
    ),
)


In [24]:
class WorkflowInput(BaseModel):
    conversation_id: str
    input_as_text: str


async def run_workflow(workflow_input: WorkflowInput) -> dict:
    with trace("Personal Trainer Agent"):
        conversation_history = prepare_conversation_for_runner(
            list(get_history(workflow_input.conversation_id))
        )
        conversation_history.append(
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": workflow_input.input_as_text}
                ],
            }
        )

        result = await Runner.run(
            personal_trainer_agent,
            input=conversation_history,
            run_config=RunConfig(
                trace_metadata={
                    "__trace_source__": "agent-builder",
                    "workflow_id": (
                        "wf_69fe0fdbffd881908e0190f1ad0dbdbb0d155924d8e1f831"
                    ),
                }
            ),
        )
        conversation_history.extend(
            [_json_safe(item.to_input_item()) for item in result.new_items]
        )
        save_history(workflow_input.conversation_id, conversation_history)
        return {
            "conversation_id": workflow_input.conversation_id,
            "output_text": result.final_output_as(str),
        }


## Session: set or resume `CONVERSATION_ID`

- **`None` + `RESUME_LAST_IF_UNSET = True` (default):** continues `last_active_conversation_id` from the JSON file when present.
- **`None` + `RESUME_LAST_IF_UNSET = False`:** starts a brand-new conversation id.
- **Explicit uuid:** use that thread if it exists in the file; otherwise a new id is created.

All turns for the chosen id are stored under `conversations` in the same JSON file.

In [25]:
CONVERSATION_ID: str | None = None
RESUME_LAST_IF_UNSET = True

CONVERSATION_ID = get_or_create_conversation(
    CONVERSATION_ID, resume_last_if_unset=RESUME_LAST_IF_UNSET
)
print(f"Using HISTORY_FILE={HISTORY_FILE}")
print(f"CONVERSATION_ID={CONVERSATION_ID}")


Resumed conversation: 1e5396ed-65a8-46fe-8d7f-336487a0b6bd
Using HISTORY_FILE=/Users/andrewallkin/personal/repos/strava-automation-service/personal_trainer_chat_history.json
CONVERSATION_ID=1e5396ed-65a8-46fe-8d7f-336487a0b6bd


## Run one turn

Re-run this cell (edit `USER_MESSAGE` first) as many times as you like. History is appended to the JSON store after each run.

**Jupyter / IPython:** this cell uses top-level `await` (the kernel already has an event loop — do not use `asyncio.run()` here).

In [27]:
USER_MESSAGE = "Do you think I am fit enough for a sub 3:45 marathon?"

out = await run_workflow(
    WorkflowInput(
        conversation_id=CONVERSATION_ID,
        input_as_text=USER_MESSAGE,
    )
)
print(out["output_text"])


Yes — I think you look fit enough to be in the sub-3:45 marathon range, with some caveats.

Observation:
- Sub-3:45 requires about 5:20/km pace.
- Your recent training shows solid endurance volume: 201 km across your last 15 runs, and weekly mileage mostly in the 50–71 km range.
- You’ve recently run:
  - 34.2 km at 5:36/km with avg HR 162
  - 22.6 km at 5:46/km with avg HR 154
  - 32.1 km at 5:?? no, not recent, but your longest-run progression has several runs in the 25–34 km range.
- Your 34.2 km run shows strong aerobic durability: weighted HR decoupling was +3.4%, which is a good sign.

Interpretation:
- That volume and long-run consistency are generally enough to support a sub-3:45 attempt.
- The biggest question is not raw endurance; it’s whether you can hold 5:20/km for the full marathon without late-race fade.
- Your recent long run pace of 5:36/km is slower than marathon goal pace, but that’s normal for a long run. The encouraging part is that you can handle the distance and 

## Optional: inspect saved history (raw JSON items)

In [ ]:
from pprint import pprint

pprint(get_history(CONVERSATION_ID))
